In [1]:
!pip install folium geopy networkx

# Stevens Campus Navigation - Initial Research

Goal: investigate Python libraries that could be used to create a
campus navigation application.

Libraries explored:
- Folium - interactive map visualization
- Geopy - coordinate/distance calculations
- NetworkX - graph-based shortest path routing
- OpenStreetMap / Nominatim - geographic location data

In [2]:
import requests

BASE_URL = "https://nominatim.openstreetmap.org/search"

headers = {
    "User-Agent": "StevensCampusNavigationStudentProject/1.0"
}

params = {
    "q": "Stevens Institute of Technology, Hoboken, NJ",
    "format": "json",
    "limit": 1
}

response = requests.get(
    BASE_URL,
    params=params,
    headers=headers
)

results = response.json()

results

[{'place_id': 357885903,
  'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
  'osm_type': 'way',
  'osm_id': 27080296,
  'lat': '40.7448096',
  'lon': '-74.0252392',
  'class': 'amenity',
  'type': 'university',
  'place_rank': 30,
  'importance': 0.47544946407940364,
  'addresstype': 'amenity',
  'name': 'Stevens Institute of Technology',
  'display_name': 'Stevens Institute of Technology, 1, Wittpenn Walk, Hoboken, Hudson County, New Jersey, 07030, United States',
  'boundingbox': ['40.7417869', '40.7479313', '-74.0282585', '-74.0228554']}]

In [4]:
latitude = float(results[0]["lat"])
longitude = float(results[0]["lon"])

print(latitude, longitude)

40.7448096 -74.0252392


In [10]:
import folium

campus_map = folium.Map(
    location=[latitude, longitude],
    zoom_start=17,
    tiles=None
)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    attr="OpenStreetMap contributors, Tiles style by Humanitarian OpenStreetMap Team",
    name="OpenStreetMap HOT"
).add_to(campus_map)

folium.Marker(
    [latitude, longitude],
    popup="Stevens Institute of Technology",
    tooltip="Stevens"
).add_to(campus_map)

campus_map

In [14]:
def find_location(place):

    params = {
        "q": place,
        "format": "json",
        "limit": 1
    }

    response = requests.get(
        BASE_URL,
        params=params,
        headers=headers
    )

    results = response.json()

    if len(results) == 0:
        return None

    return (
        float(results[0]["lat"]),
        float(results[0]["lon"])
    )

In [16]:
find_location("Babbio Center Stevens Institute Hoboken NJ")

In [17]:
locations = {
    "Babbio Center": find_location(
        "Babbio Center Stevens Institute Hoboken NJ"
    ),

    "Howe Center": find_location(
        "Howe Center Stevens Institute Hoboken NJ"
    ),

    "Gateway Academic Center": find_location(
        "Gateway Academic Center Stevens Hoboken NJ"
    )
}

locations

{'Babbio Center': None, 'Howe Center': None, 'Gateway Academic Center': None}

In [18]:
locations = {
    "Babbio Center": (40.7428, -74.0270),
    "Howe Center": (40.7448, -74.0249),
    "Gateway Academic Center": (40.7439, -74.0258)
}

locations

{'Babbio Center': (40.7428, -74.027),
 'Howe Center': (40.7448, -74.0249),
 'Gateway Academic Center': (40.7439, -74.0258)}

In [19]:
for name, coordinates in locations.items():

    folium.Marker(
        coordinates,
        popup=name,
        tooltip=name
    ).add_to(campus_map)

campus_map

In [20]:
route = [
    locations["Babbio Center"],
    locations["Gateway Academic Center"],
    locations["Howe Center"]
]

folium.PolyLine(
    route,
    weight=5,
    tooltip="Example Route"
).add_to(campus_map)

campus_map

In [21]:
from geopy.distance import geodesic
point1 = locations["Babbio Center"]
point2 = locations["Howe Center"]

if point1 is not None and point2 is not None:

    distance = geodesic(
        point1,
        point2
    ).meters

    print(
        f"Straight-line distance: {distance:.1f} meters"
    )

Straight-line distance: 284.2 meters
